# Lecture 3: Corpus Access & Lexical Resources — Answer Key
### NLP Course 2027 — Week 02

---

This notebook provides complete, worked answers for the four exercises in **Lecture 3: Corpus Access & Lexical Resources**.

In [1]:
import nltk

for pkg in ['gutenberg', 'brown', 'reuters', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.corpus import gutenberg, brown, stopwords
from nltk.corpus import wordnet as wn
from nltk import FreqDist
from collections import Counter

print('Setup complete.')

Setup complete.


---
## Exercise 3.1

**Task:** Compute the average sentence length (in words) for each Gutenberg text. Which author writes the longest sentences?

**Key concept:** *Average sentence length* is a classic stylometric feature. Longer sentences suggest more complex syntax (embedding, coordination). Shorter sentences are typical of narrative prose, dialogue-heavy fiction, or modern journalistic style. This is computable in one pass because `gutenberg.sents()` returns a list of lists.

In [2]:
from nltk.corpus import gutenberg

print(f'Gutenberg texts: {len(gutenberg.fileids())}\n')

# Compute per-text statistics
results = []
for fileid in gutenberg.fileids():
    sents = gutenberg.sents(fileid)       # list of lists of tokens
    words = gutenberg.words(fileid)       # flat list of all tokens

    num_sents = len(sents)
    num_words = len(words)
    # Average sentence length in words (including punctuation tokens)
    avg_sent_len = num_words / num_sents if num_sents > 0 else 0

    # Derive a readable author name from the filename
    # Format: 'author-title.txt'  →  'Author'
    author = fileid.split('-')[0].capitalize()

    results.append({
        'fileid': fileid,
        'author': author,
        'sentences': num_sents,
        'words': num_words,
        'avg_sent_len': avg_sent_len,
    })

# Sort by average sentence length descending
results.sort(key=lambda r: r['avg_sent_len'], reverse=True)

print(f'{"File":<32} {"Author":<12} {"Sentences":>10} {"Words":>8} {"Avg sent len":>14}')
print('-' * 82)
for r in results:
    print(f'{r["fileid"]:<32} {r["author"]:<12} {r["sentences"]:>10,} {r["words"]:>8,} {r["avg_sent_len"]:>14.1f}')

print()
print(f'Longest average sentence: {results[0]["fileid"]} ({results[0]["avg_sent_len"]:.1f} words/sentence)')

Gutenberg texts: 18

File                             Author        Sentences    Words   Avg sent len
----------------------------------------------------------------------------------
milton-paradise.txt              Milton            1,851   96,825           52.3
whitman-leaves.txt               Whitman           4,250  154,883           36.4
bible-kjv.txt                    Bible            30,103 1,010,654           33.6
austen-sense.txt                 Austen            4,999  141,576           28.3
austen-persuasion.txt            Austen            3,747   98,171           26.2
melville-moby_dick.txt           Melville         10,059  260,819           25.9
austen-emma.txt                  Austen            7,752  192,427           24.8
chesterton-brown.txt             Chesterton        3,806   86,063           22.6
edgeworth-parents.txt            Edgeworth        10,230  210,663           20.6
chesterton-ball.txt              Chesterton        4,779   96,996           20.3
carr

**Expected result:** Milton's *Paradise Lost* (`milton-paradise.txt`) typically has the longest average sentence length (~52 words/sentence), reflecting its epic blank-verse style with elaborate syntactic embedding. In contrast, the Bible texts have shorter average sentences due to the verse structure. Jane Austen's novels sit in the middle range.

**Caveat:** NLTK's sentence segmenter treats poetic line breaks as sentence boundaries in verse texts, so sentence counts for poetry are inflated and average lengths are deflated.

---
## Exercise 3.2

**Task:** Using the Brown corpus, compare the 20 most common words in the `'news'` vs `'romance'` categories.

**Key concept:** The Brown corpus is deliberately *balanced* across 15 genres, making genre comparison meaningful. Function-word distributions differ across registers (news uses `said`, `government`; romance uses `she`, `her`, `love`). This is the foundation of *authorship attribution* and *genre classification*.

In [3]:
from nltk.corpus import brown, stopwords
from nltk import FreqDist

# Get lowercased tokens for each category
news_words    = [w.lower() for w in brown.words(categories='news')    if w.isalpha()]
romance_words = [w.lower() for w in brown.words(categories='romance') if w.isalpha()]

news_fdist    = FreqDist(news_words)
romance_fdist = FreqDist(romance_words)

news_top20    = [word for word, _ in news_fdist.most_common(20)]
romance_top20 = [word for word, _ in romance_fdist.most_common(20)]

# Side-by-side comparison
print(f'{"Rank":<6} {"News word":<20} {"Count":>6}   {"Romance word":<20} {"Count":>6}')
print('-' * 65)
for i, (nw, rw) in enumerate(zip(news_top20, romance_top20), 1):
    print(f'{i:<6} {nw:<20} {news_fdist[nw]:>6,}   {rw:<20} {romance_fdist[rw]:>6,}')

# Words unique to each top-20
news_unique    = set(news_top20)    - set(romance_top20)
romance_unique = set(romance_top20) - set(news_top20)
shared         = set(news_top20)    & set(romance_top20)

print(f'\nShared in both top-20   : {sorted(shared)}')
print(f'Unique to news top-20   : {sorted(news_unique)}')
print(f'Unique to romance top-20: {sorted(romance_unique)}')

Rank   News word             Count   Romance word          Count
-----------------------------------------------------------------
1      the                   6,386   the                   2,988
2      of                    2,861   and                   1,905
3      and                   2,186   to                    1,517
4      to                    2,144   a                     1,383
5      a                     2,130   of                    1,202
6      in                    2,020   he                    1,068
7      for                     969   was                     999
8      that                    829   i                       951
9      is                      733   in                      930
10     was                     717   she                     728
11     on                      691   it                      717
12     he                      642   had                     695
13     at                      636   her                     680
14     with             

**Expected observations:**

- Both lists are dominated by function words (`the`, `a`, `of`, `in`, `to`) — these are shared and tell us little about topic.
- **News-unique** top-20 words: `said`, `percent`, `mr`, `year`, `new`, `state` — clearly attributive and factual vocabulary.
- **Romance-unique** top-20 words: `she`, `her`, `had`, `could`, `would`, `back` — personal pronouns and modal/past verbs suggest interpersonal narrative.
- This confirms that even simple frequency analysis can distinguish genres; a classifier trained on these distributions would be highly accurate.

---
## Exercise 3.3

**Task:** In WordNet, find all hyponyms of `'vehicle'`. How many are there? Use `.closure(lambda s: s.hyponyms())`.

**Key concept:** *Hyponyms* are more-specific concepts (vehicle → car → sedan). `.closure()` performs a transitive walk down the WordNet hierarchy, collecting all descendants at every depth — much more powerful than just calling `.hyponyms()` once (which only goes one level).

In [4]:
from nltk.corpus import wordnet as wn

# Inspect available senses for 'vehicle'
vehicle_synsets = wn.synsets('vehicle')
print('Synsets for "vehicle":')
for s in vehicle_synsets:
    print(f'  {s.name()}: {s.definition()}')
print()

# Use the primary (most common) noun sense
vehicle = wn.synset('vehicle.n.01')

# .closure() performs a breadth-first transitive walk through the given relation.
# Each call to the lambda returns the next level's synsets.
all_hyponyms = list(vehicle.closure(lambda s: s.hyponyms()))

print(f'Total hyponyms of vehicle.n.01 (all depths): {len(all_hyponyms)}')
print()

# Direct hyponyms (depth 1 only)
direct = vehicle.hyponyms()
print(f'Direct hyponyms (depth 1): {len(direct)}')
for s in direct[:10]:
    print(f'  {s.name()}: {s.definition()}')
print('  ...')

# Show some leaf-level (deepest) hyponyms to illustrate hierarchy depth
print()
print('Sample of all hyponyms (every 10th):')
for s in all_hyponyms[::10]:
    print(f'  {s.name()}: {s.definition()[:70]}')

Synsets for "vehicle":
  vehicle.n.01: a conveyance that transports people or objects
  vehicle.n.02: a medium for the expression or achievement of something
  vehicle.n.03: any substance that facilitates the use of a drug or pigment or other material that is mixed with it
  fomite.n.01: any inanimate object (as a towel or money or clothing or dishes or books or toys etc.) that can transmit infectious agents from one person to another

Total hyponyms of vehicle.n.01 (all depths): 519

Direct hyponyms (depth 1): 8
  wheeled_vehicle.n.01: a vehicle that moves on wheels and usually has a container for transporting things or people
  rocket.n.01: any vehicle self-propelled by a rocket engine
  craft.n.02: a vehicle designed for navigation in or on water or air or through outer space
  bumper_car.n.01: a small low-powered electrically powered vehicle driven on a special platform where there are many others to be dodged
  steamroller.n.02: vehicle equipped with heavy wide smooth rollers for 

**Key points:**

- `vehicle.hyponyms()` (no closure) returns only the immediate children — typically around 10–15 synsets.
- With `.closure()`, the walk recurses all the way down: car → sedan → compact sedan, giving **hundreds** of descendants.
- This transitive closure is exactly what is used internally when computing `path_similarity` — the path length between two synsets travels up through their common ancestor.
- The `lambda s: s.hyponyms()` is a *relation function* — you can substitute `s.hypernyms()` to walk *up* the tree instead.

---
## Exercise 3.4

**Task:** Write a function `find_antonyms(word)` that returns all antonyms using WordNet lemmas.

**Key concept:** In WordNet, antonymy is a *lexical* relation (between *lemmas*) not a *semantic* one (between *synsets*). You must drill down to the lemma level. Each synset contains multiple lemmas; each lemma can have `.antonyms()` returning its lexical opposites.

In [5]:
from nltk.corpus import wordnet as wn


def find_antonyms(word: str) -> dict:
    """
    Find all antonyms of a word via WordNet lemmas.

    WordNet stores antonymy as a *lexical* relation between lemmas,
    not as a semantic relation between synsets.  To find antonyms we:
       1. Get all synsets for the word.
       2. For each synset, iterate its lemmas.
       3. For each lemma, call .antonyms() and collect the result.

    Returns
    -------
    dict mapping each sense (synset name) to a list of antonym strings.
    Only senses with at least one antonym are included.
    """
    antonym_map = {}

    for synset in wn.synsets(word):
        sense_antonyms = []
        for lemma in synset.lemmas():
            # lemma.antonyms() returns a list of Lemma objects
            for ant in lemma.antonyms():
                sense_antonyms.append(ant.name().replace('_', ' '))
        if sense_antonyms:
            antonym_map[synset.name()] = {
                'definition': synset.definition(),
                'antonyms': sorted(set(sense_antonyms))
            }

    return antonym_map


# ── Test on a range of word classes ─────────────────────────────────────────
test_words = ['good', 'fast', 'love', 'happy', 'increase', 'hot', 'north']

for word in test_words:
    result = find_antonyms(word)
    if result:
        print(f'Antonyms of "{word}":')
        for sense, info in result.items():
            print(f'  [{sense}] "{info["definition"][:55]}"')
            print(f'    antonyms: {info["antonyms"]}')
    else:
        print(f'No direct antonyms found for "{word}" in WordNet.')
    print()

Antonyms of "good":
  [good.n.02] "moral excellence or admirableness"
    antonyms: ['evil', 'evilness']
  [good.n.03] "that which is pleasing or valuable or useful"
    antonyms: ['bad', 'badness']
  [good.a.01] "having desirable or positive qualities especially those"
    antonyms: ['bad']
  [good.a.03] "morally admirable"
    antonyms: ['evil']
  [well.r.01] "(often used as a combining form) in a good or proper or"
    antonyms: ['ill']

Antonyms of "fast":
  [fast.a.01] "acting or moving or capable of acting or moving quickly"
    antonyms: ['slow']
  [fast.a.02] "(used of timepieces) indicating a time ahead of or late"
    antonyms: ['slow']
  [fast.a.03] "at a rapid tempo"
    antonyms: ['slow']

Antonyms of "love":
  [love.n.01] "a strong positive emotion of regard and affection"
    antonyms: ['hate']
  [love.v.01] "have a great affection or liking for"
    antonyms: ['hate']

Antonyms of "happy":
  [happy.a.01] "enjoying or showing or marked by joy or pleasure"
    antonyms: [

**Notes on the design:**

- `ant.name().replace('_', ' ')` converts multi-word lemma names (e.g. `'ill_health'`) to readable strings.
- `sorted(set(...))` deduplicates antonyms that appear via multiple lemmas of the same synset.
- Not all words have antonyms in WordNet. WordNet only encodes antonyms for the *most frequently used* lemmas and focuses on *direct* antonyms (not mere contraries).
- `'love'` may return no direct antonym because WordNet treats `hate` as a co-hyponym of an emotion synset rather than a direct lexical antonym — this is a known gap.

| Antonym type | WordNet coverage |
|---|---|
| Gradable adjectives (`good`/`bad`) | Excellent |
| Verb opposites (`increase`/`decrease`) | Good |
| Directional (`north`/`south`) | Good |
| Emotional nouns (`love`/`hate`) | Sparse |

---
## Summary of Key Concepts

| Exercise | Concept | Key Takeaway |
|----------|---------|-------------|
| 3.1 | Corpus statistics | Average sentence length reveals syntactic complexity and authorial style |
| 3.2 | Genre comparison with FreqDist | Function-word distributions separate genres even without topic words |
| 3.3 | WordNet hierarchy + `.closure()` | Transitive hyponym closure traverses the entire IS-A tree below a concept |
| 3.4 | WordNet antonymy via lemmas | Antonymy is lexical (lemma-level), not semantic (synset-level) |

---
*NLP Course 2027 — Week 02*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**